In [1]:

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

from lab2_utils import load_data, clean_data, make_target, get_split


# 1. Load, clean and remove outliers
df = clean_data(load_data())
df = df[df["Gr Liv Area"] <= 4000].copy()

# 2. Create the three selected engineered features
df["HouseAge"] = (df["Yr Sold"] - df["Year Built"]).clip(lower=0)
df["TotalArea"] = df["Gr Liv Area"] + df["Total Bsmt SF"]
df["TotalBathrooms"] = (
    df["Full Bath"] + df["Bsmt Full Bath"]
    + 0.5 * (df["Half Bath"] + df["Bsmt Half Bath"])
)

# 3. Create target and remove identifiers
y = make_target(df)
X = df.drop(columns=["Order", "PID", "SalePrice"])

# MS SubClass is categorical, despite being numeric
X["MS SubClass"] = X["MS SubClass"].astype(str)

# 4. Ordinal encoding for genuinely ordered categories
quality = {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

ordinal_mappings = {
    **{col: quality for col in [
        "Exter Qual", "Exter Cond",
        "Bsmt Qual", "Bsmt Cond",
        "Heating QC", "Kitchen Qual",
        "Fireplace Qu", "Garage Qual",
        "Garage Cond", "Pool QC"
    ]},
    "Bsmt Exposure": {"NA": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFin Type 1": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFin Type 2": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "Garage Finish": {"NA": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Lot Shape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "Land Slope": {"Sev": 0, "Mod": 1, "Gtl": 2},
    "Paved Drive": {"N": 0, "P": 1, "Y": 2},
    "Functional": {"NA": 0, "Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4,
                   "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "Utilities": {"ELO": 0, "NoSeWa": 1, "NoSewr": 2, "AllPub": 3}
}

for col, mapping in ordinal_mappings.items():
    if col in X.columns:
        unknown = set(X[col].dropna().unique()) - set(mapping)
        if unknown:
            raise ValueError(f"Unexpected values in {col}: {unknown}")
        X[col] = X[col].map(mapping)

# 5. Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = get_split(X, y)

# 6. Separate numerical/ordinal and nominal features
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
nominal_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# Median imputation for numerical and ordinal features
# Most-frequent imputation + one-hot encoding for nominal features
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("nominal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), nominal_cols)
])

# 7. Create the XGBoost pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(random_state=42))
])

# 8. Stratified 5-fold CV with balanced sample weights
cv5_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_final = []

for train_idx, val_idx in cv5_final.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    weights = compute_sample_weight("balanced", y_fold_train)
    fold_model = clone(model_pipeline)

    fold_model.fit(
        X_fold_train, y_fold_train,
        classifier__sample_weight=weights
    )

    predictions = fold_model.predict(X_fold_val)
    cv_scores_final.append(
        f1_score(y_fold_val, predictions, average="macro")
    )

print("CV fold scores:", np.round(cv_scores_final, 4))
print(f"Mean CV Macro-F1: {np.mean(cv_scores_final):.4f}")
print(f"Std CV Macro-F1: {np.std(cv_scores_final):.4f}")

# 9. Fit the initial model on the complete training set
final_model = clone(model_pipeline)
final_weights = compute_sample_weight("balanced", y_train)

final_model.fit(
    X_train, y_train,
    classifier__sample_weight=final_weights
)

print("\nInitial model trained successfully.")
print("Training houses:", len(X_train))
print("Test houses:", len(X_test))
print("Features include the three selected engineered features.")


CV fold scores: [0.7216 0.7889 0.7718 0.7231 0.7718]
Mean CV Macro-F1: 0.7554
Std CV Macro-F1: 0.0277

Initial model trained successfully.
Training houses: 2340
Test houses: 585
Features include the three selected engineered features.


In [2]:
# Evaluating my selected configuration on the test set
y_test_pred = final_model.predict(X_test)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

print(f"Test Macro-F1 (before tuning): {test_macro_f1:.4f}")

Test Macro-F1 (before tuning): 0.7452


Test macro-F1: 0.748 - baseline model

Test Macro-F1 (before tuning): 0.7452 - model using strategies